# Module 2 : Réseau de Neurones Convolutif (CNN) — MNIST & Embeddings 🔍

**Fondations Deep Learning | Papa Séga WADE — [papasegawade.com](https://papasegawade.com)**

---

## 🎯 Objectifs

- Comprendre pourquoi le CNN surpasse le DNN sur les images
- Maîtriser les couches **Conv2D**, **MaxPooling**, **BatchNorm** et **Dropout**
- Atteindre **~99%** de précision sur MNIST
- Extraire et visualiser les **embeddings** en 2D (t-SNE, UMAP)

---

## 📐 Pourquoi un CNN plutôt qu'un DNN ?

| | DNN (Dense) | CNN (Convolutif) |
|--|--|--|
| Entrée | Vecteur 1D (784) | Image 2D (28×28×1) |
| Structure spatiale | ❌ Perdue (flatten) | ✅ Préservée |
| Paramètres | Beaucoup (784×512=400K) | Peu (partage des poids) |
| Performance images | ~98% | ~99%+ |

La **couche de convolution** applique un **filtre** (kernel) qui glisse sur l'image, détectant des motifs locaux (bords, courbes, textures).

## 1. Imports et préparation des données

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.manifold import TSNE

print("TensorFlow:", tf.__version__)

# Chargement
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Normalisation + ajout de la dimension de canal (28,28) → (28,28,1)
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0
X_train = np.expand_dims(X_train, -1)
X_test  = np.expand_dims(X_test, -1)

y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat  = keras.utils.to_categorical(y_test, 10)

print(f"Shape images CNN : {X_train.shape}")

## 2. Architecture du CNN

```
Input (28×28×1)
    ↓
Conv2D (32 filtres, 3×3) + BatchNorm + ReLU
    ↓
MaxPooling2D (2×2)
    ↓
Conv2D (64 filtres, 3×3) + BatchNorm + ReLU
    ↓
MaxPooling2D (2×2)
    ↓
Conv2D (128 filtres, 3×3) + ReLU
    ↓
Flatten → Dense (128) + ReLU  ← Couche d'EMBEDDINGS 📍
    ↓
Dropout (0.5)
    ↓
Dense (10) + Softmax
```

In [ ]:
inputs = keras.Input(shape=(28, 28, 1))

x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(inputs)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2,2)(x)

x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2,2)(x)

x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = layers.Flatten()(x)

# Couche d'embeddings (représentation vectorielle)
embeddings = layers.Dense(128, activation='relu', name='embeddings')(x)
x = layers.Dropout(0.5)(embeddings)
outputs = layers.Dense(10, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)
model.summary()

## 3. Entraînement

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\n🏆 Test Accuracy : {test_acc*100:.2f}%")

## 4. Visualisation des Embeddings avec t-SNE

On extrait la sortie de la couche `embeddings` (vecteurs de 128 dimensions) et on les réduit à 2D avec **t-SNE** pour visualiser comment le réseau organise les classes.

In [ ]:
# Créer un modèle qui renvoie les embeddings
embedding_model = Model(inputs=model.input,
                        outputs=model.get_layer('embeddings').output)

N = 3000  # Prendre un sous-ensemble pour accélérer
X_sample = X_test[:N]
y_sample = y_test[:N]

embeds = embedding_model.predict(X_sample, verbose=0)
print(f"Shape des embeddings : {embeds.shape}")

# Réduction t-SNE 128D → 2D
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
embeds_2d = tsne.fit_transform(embeds)
print("t-SNE terminé !")

In [ ]:
# Visualisation
colors = plt.cm.tab10(np.linspace(0, 1, 10))

fig, ax = plt.subplots(figsize=(12, 10))
for digit in range(10):
    mask = y_sample == digit
    ax.scatter(embeds_2d[mask, 0], embeds_2d[mask, 1],
               c=[colors[digit]], label=str(digit), s=15, alpha=0.7)

ax.legend(title='Chiffre', fontsize=10)
ax.set_title('Visualisation t-SNE des Embeddings CNN — MNIST (3000 exemples)', fontsize=14)
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')
plt.tight_layout()
plt.show()

---

## ✅ Bilan du Module 2

| Concept | Ce que vous avez appris |
|---------|------------------------|
| Conv2D | Filtre glissant → détecte motifs locaux |
| MaxPooling | Réduit la dimension spatiale |
| BatchNorm | Stabilise l'entraînement |
| Embeddings | Représentation vectorielle interne apprise |
| t-SNE | Visualiser des données de haute dimension |

➡️ **Module suivant :** Reconstruire ce modèle avec **PyTorch** !